# Lección 15.3 — Reentrenamiento y versionado de datos y modelos

> **Módulo 15 · ML en producción y responsabilidad** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 15.1 y 15.2

### 🎯 Objetivos
Al terminar esta lección podrás:
- Comparar estrategias de **reentrenamiento**: nunca, periódico, disparado por alertas; con ventana **deslizante** o **creciente**.
- Decidir si un modelo nuevo reemplaza al actual con el esquema **campeón vs. retador** (*champion/challenger*).
- Versionar **datos** (huellas o *hashes*) y **modelos** (registro) para poder reproducir y volver atrás.
- Explicar los despliegues en **sombra**, **canario** y la **vuelta atrás** (*rollback*).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

plt.rcParams["figure.figsize"] = (9, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el médico que se actualiza 🩺

Un médico que se graduó hace 30 años y **nunca** volvió a estudiar es peligroso: la medicina cambió. Pero un médico que cambia de tratamiento **cada vez que lee un artículo nuevo** también lo es: se deja llevar por cada estudio pequeño y ruidoso.

El buen médico se actualiza **con criterio**:
- con una **frecuencia razonable** (congresos, cursos);
- dando **más peso a la evidencia reciente**, sin olvidar lo que sigue siendo válido;
- **comparando** el tratamiento nuevo con el actual antes de cambiar;
- y llevando un **historial** de qué recetó, cuándo y por qué.

Eso es exactamente reentrenar y versionar un modelo.

### ✋ Ejercicio 1 — Calentamiento
En la Lección 15.2 viste que, tras la oferta de un competidor, el modelo de cancelación empeoró. Un compañero propone: *"Reentrenemos el modelo **todos los días** con los datos del día anterior, así nunca se queda viejo"*. Escribe una ventaja y dos riesgos.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Ventaja:** se adapta muy rápido a los cambios.

**Riesgos:**
1. Con los datos de **un solo día** hay pocos casos: el modelo aprende **ruido** (un día raro, un feriado) y cambia sin parar.
2. Si un día llegan **datos erróneos** (un error de sistema), el modelo nuevo los aprende y se despliega automáticamente sin que nadie lo revise.
3. Además: las etiquetas de cancelación llegan con **retraso** (Lección 15.2), así que "los datos de ayer" ni siquiera tienen la respuesta todavía; y cada versión nueva hay que validarla, registrarla y vigilarla.
</details>

---
## 2. 📖 El concepto: estrategias de reentrenamiento

Dos decisiones independientes:

**¿Cuándo reentrenar?**

| Estrategia | Cuándo | Ventaja | Riesgo |
|---|---|---|---|
| **Nunca** (modelo estático) | — | Simple, estable | Envejece |
| **Periódica** | Cada semana / mes / trimestre | Predecible, fácil de operar | Reentrena aunque no haga falta (y cuesta) |
| **Disparada** | Cuando el monitoreo da una alerta (Lección 15.2) | Solo cuando hace falta | Depende de que las alertas funcionen |
| **Continua / en línea** | Con cada dato nuevo | Máxima adaptación | Inestable, difícil de validar |

**¿Con qué datos?**

```
Ventana CRECIENTE:   [mes 1 ............................. mes 12]   todo el historial
Ventana DESLIZANTE:                       [mes 10 .. mes 12]          solo lo reciente
```

- **Creciente:** más datos, más estable… pero mezcla el mundo de antes con el de ahora.
- **Deslizante:** se parece al mundo actual… pero con menos datos y olvidando lo antiguo.

▶️ Vamos a simular **24 meses** de producción del modelo de cancelación, con dos cambios del mundo:
- **Meses 9–14:** el contrato anual retiene cada vez menos (cambio **gradual**, como en la Lección 15.2).
- **Mes 18:** una subida de tarifas hace a los clientes **mucho más sensibles al precio** (cambio **repentino**).

Las etiquetas de cada mes llegan al terminar el mes, así que el modelo del mes *m* solo puede usar datos hasta el mes *m − 1*.

In [ ]:
COLUMNAS = ["meses_cliente", "llamadas_soporte", "cargo_mensual", "contrato_anual"]

def efectos_del_mundo(mes):
    efecto_contrato = -1.8 if mes <= 8 else max(-0.2, -1.8 + 0.27 * (mes - 8))   # cambio gradual
    efecto_precio = 0.002 if mes < 18 else 0.0045                                  # cambio repentino
    return efecto_contrato, efecto_precio

def generar_mes(mes, n=1500):
    rng = np.random.default_rng(1000 + mes)
    df = pd.DataFrame({"meses_cliente": rng.integers(1, 72, n), "llamadas_soporte": rng.poisson(1.5, n),
                       "cargo_mensual": rng.integers(200, 1500, n), "contrato_anual": rng.integers(0, 2, n)})
    contrato, precio = efectos_del_mundo(mes)
    riesgo = (-1.5 + 0.9 * df.llamadas_soporte - 0.05 * df.meses_cliente
              + precio * df.cargo_mensual + contrato * df.contrato_anual - (precio - 0.002) * 800)
    df["cancelo"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)
    df["mes"] = mes
    return df

DATOS = pd.concat([generar_mes(m) for m in range(1, 25)], ignore_index=True)

def entrenar(meses):
    d = DATOS[DATOS.mes.isin(list(meses))]
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(d[COLUMNAS], d.cancelo)

def auc_del_mes(modelo, mes):
    d = DATOS[DATOS.mes == mes]
    return roc_auc_score(d.cancelo, modelo.predict_proba(d[COLUMNAS])[:, 1])

print(DATOS.groupby("mes").cancelo.mean().round(2).to_dict())

### 🔮 Ejercicio 2 — Antes de simular
Compararemos tres estrategias, todas empezando con un modelo entrenado en los meses 1–3:
- **Nunca** reentrenar.
- **Mensual con ventana creciente** (todo el historial).
- **Mensual con ventana deslizante** de 3 meses.

¿Cuál crees que tendrá el mejor AUC medio en los meses 4–24? ¿Cuál sufrirá más tras el mes 18?

✍️ *Mi predicción:*

In [ ]:
def simular(estrategia, ventana=3, umbral=0.03):
    modelo = entrenar([1, 2, 3])
    referencia = np.mean([auc_del_mes(modelo, m) for m in [1, 2, 3]])      # AUC "de fábrica"
    aucs, reentrenos = [], 0
    for mes in range(4, 25):
        if mes > 4:
            if estrategia == "creciente":
                modelo = entrenar(range(1, mes)); reentrenos += 1
            elif estrategia == "deslizante":
                modelo = entrenar(range(max(1, mes - ventana), mes)); reentrenos += 1
            elif estrategia == "disparada" and auc_del_mes(modelo, mes - 1) < referencia - umbral:
                meses_nuevos = range(max(1, mes - ventana), mes)                      # 🚨 alerta: reentrenar
                modelo = entrenar(meses_nuevos); reentrenos += 1
                referencia = np.mean([auc_del_mes(modelo, m) for m in meses_nuevos])
        aucs.append(auc_del_mes(modelo, mes))
    return np.array(aucs), reentrenos

resultados = {}
for estrategia in ["nunca", "creciente", "deslizante"]:
    aucs, n = simular(estrategia)
    resultados[estrategia] = aucs
    print(f"{estrategia:<11} → AUC medio meses 4–24: {aucs.mean():.3f} · tras el mes 18: {aucs[-7:].mean():.3f} · reentrenos: {n}")

for nombre, aucs in resultados.items():
    plt.plot(range(4, 25), aucs, "o-", ms=3, label=nombre)
for x, texto in [(8.5, "empieza cambio gradual"), (17.5, "cambio repentino")]:
    plt.axvline(x, c="gray", ls=":"); plt.text(x + 0.2, 0.70, texto, fontsize=8)
plt.xlabel("Mes"); plt.ylabel("AUC del mes"); plt.legend(); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Nunca:** AUC medio ≈ **0.78** y cae a ≈ **0.72** tras el mes 18. Envejece con cada cambio.
- **Creciente:** ≈ **0.82**. Se recupera algo, pero lentamente: los datos viejos (cuando el contrato retenía y los clientes eran poco sensibles al precio) "pesan" cada vez más… y nunca se van.
- **Deslizante (3 meses):** ≈ **0.85**, la mejor. Tras el mes 18 promedia ≈ **0.88**: en 3 meses ya solo ve el mundo nuevo.

En un mundo que **cambia**, la memoria larga estorba. (Ojo: si el mundo fuera estable, la ventana creciente tendría más datos y solía ser igual o mejor.)
</details>

---
## 3. 📖 ¿Reentrenar siempre o solo cuando hace falta?

Reentrenar cada mes funciona, pero tiene un costo: cada modelo nuevo hay que validarlo, registrarlo, desplegarlo y vigilarlo. La estrategia **disparada** solo reentrena cuando el AUC del último mes cae más de un **umbral** respecto a lo esperado.

### 🔮 Ejercicio 3 — Disparada vs. mensual
¿Cuántas veces crees que reentrenará la estrategia disparada en 21 meses (umbral de 0.03)? ¿Qué AUC medio tendrá comparado con la deslizante mensual (≈ 0.85)?

✍️ *Mi predicción:*

In [ ]:
UMBRAL = 0.03      # ✏️ prueba 0.01, 0.03 y 0.06

aucs_disp, n_disp = simular("disparada", umbral=UMBRAL)
print(f"Disparada (umbral {UMBRAL}) → AUC medio {aucs_disp.mean():.3f} · reentrenos: {n_disp}")
print(f"Deslizante mensual         → AUC medio {resultados['deslizante'].mean():.3f} · reentrenos: 20")
print("AUC mes a mes (disparada):", aucs_disp.round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

Con umbral 0.03 reentrena solo **3 veces** y logra ≈ **0.84**, muy cerca de la mensual (≈ 0.85) con **una séptima parte** de los reentrenamientos.

Fíjate en el detalle mes a mes (el primer valor es el mes 4): reentrena en los meses 10 y 11 (tras el cambio gradual) y en el **22**. Tras el cambio de tarifas del mes 18, el AUC baja poco a poco (≈ 0.81, 0.82, 0.80, 0.78) y la alerta solo salta cuando la caída **supera el umbral**; entonces reentrena y el AUC salta a ≈ 0.90. Reacciona **tarde**: siempre necesita ver la caída, y solo si es lo bastante grande.

✏️ Con umbral **0.01** reentrena 5 veces (AUC ≈ 0.843); con **0.06**, solo 2 (≈ 0.836): tarda más en reaccionar y pasa más meses con un modelo degradado. El umbral es una perilla de **costo vs. calidad**.
</details>

### ✏️ Ejercicio 4 — El tamaño de la ventana deslizante
Prueba ventanas de **1, 3, 6 y 12** meses. ¿Cuál gana aquí? ¿Por qué crees que no siempre gana la más corta?

✍️ *Mis observaciones:*

In [ ]:
for ventana in [1, 3, 6, 12]:      # ✏️
    aucs, _ = simular("deslizante", ventana=ventana)
    print(f"ventana de {ventana:>2} meses → AUC medio {aucs.mean():.3f} · tras el mes 18: {aucs[-7:].mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Aquí ganan las ventanas cortas (1 y 3 meses ≈ 0.85) y la de 12 queda atrás (≈ 0.84 en promedio y ≈ 0.86 tras el mes 18 frente a ≈ 0.89). En este mundo **cambia mucho** y el modelo es **simple** (4 variables, una regresión logística): con 1,500 clientes por mes le sobran datos.

Pero no siempre es así: con **menos datos por mes** o un modelo **más complejo** (un ensamble de árboles con 100 variables), la ventana de 1 mes sería ruidosa e inestable, y en un mundo **estable** las ventanas largas ganan por tener más datos. Una solución intermedia muy usada: usar todo el historial pero **dar más peso a lo reciente** (`sample_weight` en `fit`).
</details>

---
## 4. 📖 Campeón vs. retador: no cambies a ciegas

Reentrenar no garantiza mejorar: el modelo nuevo podría haber aprendido de datos malos o de un mes raro. Por eso, antes de reemplazar, se comparan los dos en **los mismos datos recientes** que **ninguno** usó para entrenar:

```
          ┌── CAMPEÓN (en producción) ──┐
datos ────┤                             ├──► comparar en datos recientes no vistos
recientes └── RETADOR (recién entrenado)┘     ¿el retador gana por un margen claro?
                                                  ├── sí → registrar y desplegar (gradualmente)
                                                  └── no → el campeón se queda
```

### 🔮 Ejercicio 5 — ¿Quién gana?
Es el **mes 20**. El campeón se entrenó con los meses 1–3. El retador, con los meses 16–18. Los comparamos en el **mes 19** (que ninguno vio). ¿Quién gana? ¿Y si el retador se hubiera entrenado con meses 14–16?

✍️ *Mi predicción:*

In [ ]:
def campeon_vs_retador(campeon, retador, mes_prueba, margen=0.01):
    a_c, a_r = auc_del_mes(campeon, mes_prueba), auc_del_mes(retador, mes_prueba)
    decision = "🏆 el RETADOR reemplaza al campeón" if a_r > a_c + margen else "🛡️ el CAMPEÓN se queda"
    print(f"AUC campeón {a_c:.3f} · AUC retador {a_r:.3f} → {decision}")

campeon = entrenar([1, 2, 3])
campeon_vs_retador(campeon, entrenar([16, 17, 18]), mes_prueba=19)
campeon_vs_retador(campeon, entrenar([14, 15, 16]), mes_prueba=19)   # ✏️ prueba otras ventanas

<details><summary><b>👀 Ver respuesta</b></summary>

- Retador 16–18: gana **claramente** (≈ 0.88 vs. ≈ 0.75): ya incluye el mundo después del cambio de tarifas.
- Retador 14–16: **también** le gana al campeón (≈ 0.84 vs. ≈ 0.75), porque aprendió que el contrato ya no retiene, aunque todavía no vio el cambio de tarifas.

El **margen** (aquí 0.01) evita cambiar de modelo por diferencias que podrían ser ruido. Cambiar de modelo tiene costos (validar, desplegar, explicar a los usuarios): solo vale la pena si la mejora es clara.
</details>

### 📖 Desplegar con red de seguridad
Aunque el retador gane en la comparación, el mundo real puede sorprender. Por eso se despliega por etapas:

| Etapa | Qué pasa | Para qué |
|---|---|---|
| **Sombra** (*shadow*) | El retador recibe las mismas peticiones y predice, pero **sus predicciones no se usan** | Ver cómo se comporta con datos reales sin riesgo |
| **Canario** (*canary*) | Un % pequeño del tráfico (1–10 %) usa el retador | Detectar problemas afectando a pocos |
| **Prueba A/B** | Reparto aleatorio y medición del **impacto de negocio** | Saber si mejora lo que importa (no solo el AUC) |
| **Vuelta atrás** (*rollback*) | Volver al campeón con un clic si algo sale mal | Por eso **nunca se borra** la versión anterior |

### ✋ Ejercicio 6 — ¿Qué etapa?
1. Quieres saber si el retador **aumenta los clientes retenidos**, no solo el AUC.
2. Te preocupa que el retador sea demasiado lento para la app, pero no quieres afectar a nadie.
3. Tras desplegar al 100 %, el equipo de retención reporta llamadas a clientes absurdos.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Prueba A/B**: mide el impacto de negocio con grupos comparables (Lección 14.2: es un bandido con exploración fija).
2. **Sombra**: mides latencia y errores con tráfico real sin que nadie use sus predicciones.
3. **Vuelta atrás** inmediata al campeón y luego investigar (¿*training-serving skew*? ¿datos malos en el entrenamiento?).
</details>

---
## 5. 📖 Versionar datos y modelos: poder responder "¿qué pasó?"

Si dentro de seis meses alguien pregunta *"¿por qué el modelo rechazó a este cliente el 3 de marzo?"*, necesitas saber **qué versión** del modelo estaba activa, **con qué datos** se entrenó y poder **reproducirlo**. Para eso:

1. **Huella de los datos** (*hash*): un código corto calculado a partir del contenido de los datos. Si cambia **un solo número**, la huella cambia por completo.
2. **Semillas fijas**: mismos datos + mismo código + misma semilla → mismo modelo.
3. **Registro de modelos**: una tabla con cada versión, sus datos, sus métricas y su estado.

### 🔮 Ejercicio 7 — La huella
Calcularemos la huella de los datos de los meses 16–18. Luego **cambiaremos una sola celda** (un cliente que llamó 2 veces a soporte ahora figura con 3). ¿Cuánto se parecerán las dos huellas?

✍️ *Mi predicción:*

In [ ]:
def huella(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:16]

datos_16_18 = DATOS[DATOS.mes.isin([16, 17, 18])]
copia = datos_16_18.copy()
fila = copia.index[copia.llamadas_soporte == 2][0]
copia.loc[fila, "llamadas_soporte"] = 3                       # ¡un solo cambio!

print("Huella original :", huella(datos_16_18))
print("Huella alterada :", huella(copia))
print("Huella original otra vez:", huella(DATOS[DATOS.mes.isin([16, 17, 18])]))

<details><summary><b>👀 Ver respuesta</b></summary>

Las huellas original y alterada son **completamente distintas**, aunque solo cambió un número entre 4,500 filas. Y la original calculada otra vez da **exactamente** la misma. Por eso la huella sirve para comprobar que tienes **los mismos datos** con los que se entrenó un modelo. Herramientas como **DVC** (*Data Version Control*) hacen esto para conjuntos de datos enteros, junto con Git.
</details>

### ✋ Ejercicio 8 — ¿Es reproducible?
Entrenamos dos veces el mismo modelo con los mismos datos. ¿Darán exactamente las mismas predicciones? ¿Y un `RandomForestClassifier` sin `random_state`?

✍️ *Mi predicción:*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
d = DATOS[DATOS.mes.isin([16, 17, 18])]

m1, m2 = entrenar([16, 17, 18]), entrenar([16, 17, 18])
print("Logística, 2 entrenamientos iguales     :", np.allclose(m1.predict_proba(d[COLUMNAS]), m2.predict_proba(d[COLUMNAS])))

b1 = RandomForestClassifier(n_estimators=50).fit(d[COLUMNAS], d.cancelo)
b2 = RandomForestClassifier(n_estimators=50).fit(d[COLUMNAS], d.cancelo)
print("Bosque SIN random_state                 :", np.allclose(b1.predict_proba(d[COLUMNAS]), b2.predict_proba(d[COLUMNAS])))

b1 = RandomForestClassifier(n_estimators=50, random_state=0).fit(d[COLUMNAS], d.cancelo)
b2 = RandomForestClassifier(n_estimators=50, random_state=0).fit(d[COLUMNAS], d.cancelo)
print("Bosque CON random_state=0               :", np.allclose(b1.predict_proba(d[COLUMNAS]), b2.predict_proba(d[COLUMNAS])))

<details><summary><b>👀 Ver respuesta</b></summary>

- La regresión logística es **determinista**: mismo resultado siempre.
- El bosque **sin** `random_state` da predicciones **distintas** cada vez (usa azar para elegir muestras y variables, Lección 6.2).
- **Con** `random_state=0`, idénticas.

Regla: **fija todas las semillas** y anótalas en el registro. Sin eso, "reproducir el modelo del 3 de marzo" es imposible.
</details>

### 📖 El registro de modelos
Juntemos todo en un registro sencillo. En la práctica se usa una herramienta (MLflow, Weights & Biases, el registro de tu nube), pero la idea es esta tabla:

In [ ]:
registro = []

def registrar(version, meses, estado, nota=""):
    d = DATOS[DATOS.mes.isin(meses)]
    modelo = entrenar(meses)
    registro.append({"version": version, "meses_entrenamiento": f"{min(meses)}–{max(meses)}",
                     "huella_datos": huella(d), "filas": len(d), "semilla": "n/a (determinista)",
                     "auc_validacion": round(auc_del_mes(modelo, max(meses) + 1), 3),
                     "estado": estado, "nota": nota})

registrar("v1", [1, 2, 3], "retirado", "modelo inicial")
registrar("v2", [9, 10, 11], "retirado", "reentreno disparado: el contrato retiene menos")
registrar("v3", [16, 17, 18], "en producción", "reentreno tras cambio de tarifas")
pd.DataFrame(registro)

> 📝 `auc_validacion` usa el mes **siguiente** a la ventana de entrenamiento, que el modelo no vio.

---
## 6. 📖 Perillas del reentrenamiento

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Frecuencia (periódica) | Cada cuánto se reentrena | Se adapta antes; más costo y riesgo de aprender ruido | Más estable; envejece | La que permitan las etiquetas (p. ej., mensual) |
| Umbral de disparo | Cuánta caída tolera antes de reentrenar | Menos reentrenos, más tiempo con un modelo degradado | Reentrena casi siempre | Caída de AUC de 0.02–0.05, o PSI > 0.25 |
| Tamaño de la ventana | Cuánta historia usa | Más datos y estabilidad; mezcla mundos viejos | Más actual; más ruido | 3–12 meses (según volumen y velocidad del cambio) |
| Margen campeón/retador | Cuánto debe ganar el retador | Menos cambios, más estabilidad | Cambia por diferencias que pueden ser ruido | 0.01 de AUC, o una prueba estadística |
| % de canario | Tráfico inicial del retador | Detecta antes los problemas, afecta a más gente | Más seguro, más lento | 5 % |

## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo nuevo es peor que el viejo en producción | Se desplegó sin comparar, o se comparó en datos que el retador ya había visto | Campeón vs. retador en datos recientes **no vistos** |
| No puedes reproducir un modelo de hace meses | No se guardaron datos, huellas, semillas ni versiones de librerías | Registro de modelos + versionado de datos |
| Tras reentrenar automáticamente, todo empeora de golpe | Se reentrenó con datos corruptos (un error de sistema) | Validar los datos **antes** de reentrenar; canario; rollback |
| El modelo "persigue" cada pequeña fluctuación | Ventana demasiado corta o reentrenos demasiado frecuentes | Ventana más larga, pesos por antigüedad, umbral de disparo |
| El reentrenamiento no mejora nada | El problema no es la deriva (es un error, o falta información) | Diagnosticar antes de reentrenar (Lección 15.2, Ejercicio 9) |

### 🔮 Ejercicio 9 — Diagnóstico
Un sistema reentrena **automáticamente** cada noche y despliega el resultado sin comparar. Un martes, un error hace que todas las etiquetas de ese día se guarden como "canceló". ¿Qué pasará el miércoles? Propón dos defensas.

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

El miércoles, el modelo reentrenado habrá aprendido que **casi todos cancelan**: marcará a una enorme cantidad de clientes como "en riesgo" y el equipo de retención se saturará llamando a clientes felices.

Defensas:
1. **Validar los datos de entrenamiento** antes de reentrenar (¿la tasa de cancelación de hoy es creíble comparada con la histórica?).
2. **Campeón vs. retador** antes de desplegar: el retador "corrupto" perdería en datos recientes sanos.
3. **Canario + rollback**: aunque se cuele, solo afecta al 5 % y se vuelve atrás en minutos.
</details>

---
## 8. 📋 Ficha: el ciclo de reentrenamiento responsable

```
monitoreo (15.2) ─► ¿alerta o fecha programada?
                         │
                         ▼
     validar los datos nuevos ─► entrenar RETADOR (semilla fija, datos con huella)
                         │
                         ▼
     campeón vs. retador en datos recientes NO vistos ─► ¿gana por margen?
                         │ sí                                   │ no
                         ▼                                      ▼
     registrar versión ─► sombra ─► canario ─► 100 %      el campeón se queda
                                       │
                                  rollback si algo falla
```

## 🧠 Resumen en 5 puntos
1. Los modelos se reentrenan **periódicamente**, **por alertas** o **continuamente**; la disparada suele dar casi la misma calidad con muchos menos reentrenos.
2. La ventana **deslizante** sigue al mundo actual; la **creciente** da estabilidad. La elección depende de qué tanto cambia el mundo y de cuántos datos hay.
3. Un modelo nuevo solo reemplaza al actual si **gana en datos recientes no vistos** por un margen claro (campeón vs. retador).
4. Se despliega por etapas (**sombra → canario → A/B → 100 %**) y siempre con **vuelta atrás**.
5. **Huellas de datos, semillas y registro de modelos** permiten reproducir cualquier versión y responder "¿qué pasó?".

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** El mundo cambia rápido y tienes muchísimos datos por mes. ¿Qué ventana conviene?
- a) Creciente desde el inicio · b) Deslizante y relativamente corta · c) Da igual · d) No reentrenar

**2.** En campeón vs. retador, los dos modelos deben compararse en…
- a) Los datos de entrenamiento del retador · b) Datos recientes que ninguno de los dos usó para entrenar · c) Los datos del campeón · d) Datos simulados

**3.** Una huella (*hash*) de datos sirve para…
- a) Comprimir los datos · b) Comprobar que tienes exactamente los mismos datos con que se entrenó un modelo · c) Anonimizar · d) Entrenar más rápido

**4.** ¿Qué despliegue permite ver cómo predice un modelo con datos reales **sin afectar a nadie**?
- a) Canario · b) Sombra · c) A/B · d) Rollback

**5.** La estrategia disparada, comparada con reentrenar cada mes, suele…
- a) Ser siempre mejor · b) Lograr una calidad parecida con menos reentrenos, reaccionando un poco más tarde · c) No reentrenar nunca · d) Necesitar más datos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** 2. **b)** 3. **b)** 4. **b)** 5. **b)**
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un modelo de crédito tiene etiquetas que llegan **12 meses** después. ¿Qué implica eso para la estrategia de reentrenamiento y para el tamaño de la ventana?

**B. Caso de análisis 💼** Un regulador exige al banco poder explicar **cualquier** decisión de los últimos 5 años. ¿Qué tendría que guardar el banco de cada versión del modelo? Haz una lista.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El modelo siempre llega "tarde": lo más reciente con etiqueta tiene un año. Reentrenar cada semana no ayuda (no hay etiquetas nuevas que valgan la pena), así que se suele reentrenar con menos frecuencia (trimestral/semestral), usando **etiquetas intermedias** (atrasos de 60–90 días) como aproximación, y vigilando muy de cerca la deriva de datos para detectar cambios **antes** de que lleguen las etiquetas.

**B.** Por cada versión: el **archivo del modelo**; los **datos de entrenamiento** (o su ubicación y su huella); el **código** de preprocesamiento y entrenamiento (versión en Git); **semillas** y **versiones de librerías**; **métricas** de validación (incluidas las métricas por grupo, Lección 15.4); **fechas** en que estuvo activa; quién la **aprobó**; y, por cada decisión, la **versión** que la tomó y las **entradas** que recibió. Con eso se puede reproducir y explicar cualquier decisión (Lección 15.6).
</details>

### 9.3 🧩 Práctica integradora: "Diseña tu política de reentrenamiento"

Tu jefe te pide una política de reentrenamiento para este modelo con una restricción: **cada reentrenamiento cuesta** (validación, revisión humana, despliegue). Quiere el mejor equilibrio entre **calidad** y **número de reentrenos**.

**Paso 1 ▶️ — Define la puntuación.** Una forma simple de equilibrar: `puntuación = AUC medio − costo × reentrenos`. Con `costo = 0.002`, cada reentreno "cuesta" 0.002 de AUC.

In [ ]:
COSTO = 0.002     # ✏️ (en el Paso 4 lo cambiarás)

def puntuar(nombre, aucs, reentrenos):
    return {"política": nombre, "AUC medio": round(float(aucs.mean()), 3), "reentrenos": reentrenos,
            "puntuación": round(float(aucs.mean() - COSTO * reentrenos), 4)}

print(puntuar("ejemplo", np.array([0.85, 0.86]), 5))

**Paso 2 🔮 — Compara políticas.** Antes de ejecutar: ¿cuál crees que ganará con `COSTO = 0.002`: nunca, mensual deslizante (3 meses), disparada con umbral 0.03, o disparada con umbral 0.01?

✍️ *Mi apuesta:*

In [ ]:
filas = [puntuar("nunca", *simular("nunca")),
         puntuar("mensual deslizante (3)", *simular("deslizante", ventana=3)),
         puntuar("disparada umbral 0.01", *simular("disparada", umbral=0.01)),
         puntuar("disparada umbral 0.03", *simular("disparada", umbral=0.03)),
         puntuar("disparada umbral 0.06", *simular("disparada", umbral=0.06))]
pd.DataFrame(filas).sort_values("puntuación", ascending=False)

<details><summary><b>👀 Ver respuesta</b></summary>

Con `COSTO = 0.002` ganan las **disparadas**, prácticamente empatadas (puntuaciones ≈ 0.832–0.833): la de umbral 0.01 queda primera por una milésima (AUC ≈ 0.843 con 5 reentrenos), seguida de 0.06 (2 reentrenos) y 0.03 (3 reentrenos). La mensual tiene el mejor AUC (≈ 0.85), pero sus 20 reentrenos le restan 0.04. "Nunca" es la peor: no reentrenar sale muy caro en calidad.

Cuando varias opciones empatan, elige la más fácil de **explicar y operar**: por ejemplo, umbral 0.03.
</details>

**Paso 3 ✏️ — ¿Y si reentrenar fuera casi gratis?** Cambia `COSTO` a **0.0002** (por ejemplo, porque todo está automatizado) y vuelve a ejecutar las celdas de los Pasos 1 y 2. ¿Cambia la ganadora?

✍️ *Mi observación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con un costo diez veces menor gana la **mensual deslizante**: su mejor AUC ya compensa los 20 reentrenos. La "mejor" política depende del **costo de operar**, no solo de la calidad. Por eso automatizar bien el ciclo (Lección 15.1) permite reentrenar más a menudo.
</details>

**Paso 4 ✏️ — Ventana de la disparada.** Vuelve a `COSTO = 0.002`. La disparada usa una ventana de 3 meses al reentrenar. Prueba ventanas de 1, 3 y 6 meses con umbral 0.03.

In [ ]:
for ventana in [1, 3, 6]:      # ✏️
    print(puntuar(f"disparada 0.03, ventana {ventana}", *simular("disparada", ventana=ventana, umbral=0.03)))

<details><summary><b>👀 Ver respuesta</b></summary>

Gana la ventana de **3 meses** (puntuación ≈ 0.832). Con **1 mes** solo reentrena una vez (en el mes 10) y nunca reacciona al cambio de tarifas: su AUC baja a ≈ 0.80–0.82, pero sin cruzar el umbral de alarma, así que se queda todo el final con un modelo desactualizado (AUC medio ≈ 0.831). Con **6 meses** reentrena 4 veces, pero siempre mezclando meses de antes del cambio, y tampoco llega a recuperarse tras el mes 18 (≈ 0.825). Interacción interesante: **la ventana cambia también cuándo y si se disparan las alertas**, porque cada modelo nuevo trae su propio AUC de referencia.
</details>

**Paso 5 ✋ — Tu política, por escrito.** Escribe la política en 4 líneas, como la presentarías al jefe: cuándo se reentrena, con qué datos, cómo se decide si se despliega y qué se registra.

✍️ *Mi política:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> 1. **Cuándo:** cada mes revisamos el AUC del mes anterior con las etiquetas reales; si cae más de 0.03 respecto al AUC de validación del modelo vigente, reentrenamos. Además, reentrenamos al menos una vez por trimestre aunque no haya alertas.
> 2. **Con qué datos:** los últimos 3 meses con etiqueta, después de validar que no tengan errores (tasa de cancelación creíble, sin columnas vacías).
> 3. **Cómo se decide:** el retador debe superar al campeón por al menos 0.01 de AUC en el mes más reciente no usado para entrenar; luego 2 semanas en canario al 5 %, con vuelta atrás inmediata si algo falla.
> 4. **Qué se registra:** versión, meses de entrenamiento, huella de los datos, semillas, librerías, métricas (también por grupo) y quién aprobó el cambio.
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** Chip Huyen, *Designing Machine Learning Systems*, capítulo 9 (*Continual Learning and Test in Production*).
- 🛠️ **Herramientas:** **MLflow** (registro de modelos y experimentos), **DVC** (versionado de datos con Git), **Weights & Biases**.
- 📰 **Lectura:** Google Cloud, *MLOps: Continuous delivery and automation pipelines in machine learning* (los "niveles de madurez" de MLOps 0, 1 y 2).
- 📰 **Despliegue progresivo:** busca "canary deployment" y "shadow deployment" en la documentación de tu nube (AWS SageMaker, Google Vertex AI, Azure ML).

---
⬅️ **Anterior:** [15.2 Monitoreo: deriva de datos y de concepto](15.2-monitoreo-deriva-de-datos-y-concepto.ipynb) · ➡️ **Siguiente:** [15.4 Sesgo y equidad: cómo los modelos heredan prejuicios](15.4-sesgo-y-equidad.ipynb)